# AI-Driven Housing Affordability Forecasting in New York City

This notebook implements an AI-driven analytical framework to forecast neighborhood-level housing affordability across NYC boroughs. We integrate demographic, economic, transit, and crime data to:

1. Forecast affordability indicators by borough using multi-source data
2. Analyze affordability gaps using rent-to-income proxies
3. Predict future unaffordability and emerging gentrification patterns
4. Provide explainable insights via SHAP (XAI)

**Models Used:** XGBoost, Random Forest, LSTM  
**Explainability:** SHAP  
**Boroughs:** Manhattan, Bronx, Brooklyn, Queens


## 1. Setup & Library Imports

In [ ]:
# Install required libraries if not already installed
# !pip install xgboost shap scikit-learn pandas numpy matplotlib seaborn tensorflow

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
import xgboost as xgb

# SHAP for explainability
import shap

# Deep Learning (LSTM)
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
    from tensorflow.keras.callbacks import EarlyStopping
    TENSORFLOW_AVAILABLE = True
    print(f'TensorFlow version: {tf.__version__}')
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print('TensorFlow not available. LSTM section will be skipped.')

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']  # Blue, Red, Green, Orange
BOROUGHS = ['Manhattan', 'Bronx', 'Brooklyn', 'Queens']
BOROUGH_COLORS = dict(zip(BOROUGHS, PALETTE))

print('All libraries loaded successfully.')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

## 2. Data Loading & Preprocessing

In [ ]:
import os

# Data directory - update this path if running from a different location
DATA_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.'

def load_wide_csv(filename, value_col_name):
    """Load a wide-format CSV and melt it to long format (Borough, Year, Value)."""
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path)
    # Keep only rows matching our four boroughs
    df = df[df['Borough'].isin(BOROUGHS)].copy()
    # Identify year columns (numeric column names)
    year_cols = [c for c in df.columns if str(c).isdigit()]
    df_long = df.melt(
        id_vars=['Borough'],
        value_vars=year_cols,
        var_name='Year',
        value_name=value_col_name
    )
    df_long['Year'] = df_long['Year'].astype(int)
    df_long[value_col_name] = pd.to_numeric(df_long[value_col_name], errors='coerce')
    return df_long.sort_values(['Borough', 'Year']).reset_index(drop=True)


# Load all datasets
print('Loading datasets...')

datasets = {
    'median_income':        ('borough-medianhouseholdincome2024.csv',            'median_household_income'),
    'renter_income':        ('borough-medianhouseholdincomerenters2024.csv',     'renter_median_income'),
    'population':           ('borough-population.csv',                           'population'),
    'pop_density':          ('borough-populationdensity1000personspersquaremile.csv', 'pop_density_1000sqmi'),
    'homeownership':        ('borough-homeownershiprate.csv',                    'homeownership_rate'),
    'crime_rate':           ('borough-seriouscrimerateper1000residents.csv',     'crime_rate_per1000'),
    'car_free_commute':     ('borough-car-freecommuteofcommuters.csv',           'car_free_commute_pct'),
    'travel_time':          ('borough-meantraveltimetoworkminutes.csv',          'mean_travel_time_min'),
    'high_cost_purchase':   ('borough-higher-costhomepurchaseloansofhomepurchaseloans.csv', 'high_cost_purchase_loan_pct'),
    'high_cost_refi':       ('borough-higher-costrefinanceloansofrefinanceloans.csv',        'high_cost_refi_loan_pct'),
}

dfs = {}
for key, (fname, col) in datasets.items():
    try:
        dfs[key] = load_wide_csv(fname, col)
        print(f'  Loaded {key}: {dfs[key].shape} rows  ({dfs[key]["Year"].min()}–{dfs[key]["Year"].max()})')
    except Exception as e:
        print(f'  ERROR loading {key}: {e}')

# Static 2017 proximity data
subway_df = pd.read_csv(os.path.join(DATA_DIR, 'borough-residentialunitswithin12mileofasubwaystation.csv'))
park_df   = pd.read_csv(os.path.join(DATA_DIR, 'borough-residentialunitswithin14mileofapark.csv'))
subway_prox = dict(zip(subway_df['Borough'], subway_df['2017']))
park_prox   = dict(zip(park_df['Borough'],   park_df['2017']))

print('\nSubway proximity (2017):', subway_prox)
print('Park proximity (2017):', park_prox)

In [ ]:
# Merge all datasets on Borough + Year
from functools import reduce

df_list = list(dfs.values())
merged = reduce(
    lambda left, right: pd.merge(left, right, on=['Borough', 'Year'], how='outer'),
    df_list
)

# Add static proximity features (broadcast across all years)
merged['subway_proximity_pct'] = merged['Borough'].map(subway_prox)
merged['park_proximity_pct']   = merged['Borough'].map(park_prox)

# Filter to years with enough data coverage (2005–2023)
merged = merged[(merged['Year'] >= 2005) & (merged['Year'] <= 2023)].copy()
merged = merged.sort_values(['Borough', 'Year']).reset_index(drop=True)

print(f'Merged dataset shape: {merged.shape}')
print(f'Columns: {list(merged.columns)}')
print(f'\nYear range: {merged["Year"].min()} – {merged["Year"].max()}')
merged.head(10)

In [ ]:
# Feature Engineering
# ─────────────────────────────────────────────────────────────────────────────
# 1. Affordability Ratio = Renter median income / Median household income
#    Lower ratio → renters earn less relative to overall population → more vulnerable
merged['renter_income_ratio'] = merged['renter_median_income'] / merged['median_household_income']

# 2. Housing Cost Pressure Index: composite of high-cost loan percentages
#    Higher → more households taking expensive loans → less affordable market
merged['cost_pressure_index'] = (
    merged['high_cost_purchase_loan_pct'].fillna(0) +
    merged['high_cost_refi_loan_pct'].fillna(0)
) / 2

# 3. Affordability Stress Index (primary target)
#    Combines cost pressure with income vulnerability
#    Higher → more stressed (less affordable)
merged['affordability_stress'] = (
    merged['cost_pressure_index'] /
    (merged['renter_income_ratio'].clip(lower=0.01))
)

# 4. Year-over-year income growth (per borough)
merged['income_growth_yoy'] = merged.groupby('Borough')['median_household_income'].pct_change()
merged['renter_income_growth_yoy'] = merged.groupby('Borough')['renter_median_income'].pct_change()

# 5. Income divergence: gap between overall and renter incomes (in $)
merged['income_gap'] = merged['median_household_income'] - merged['renter_median_income']

# 6. Renter share proxy: 1 - homeownership rate
merged['renter_share'] = 1 - merged['homeownership_rate']

# 7. Borough encoding
borough_map = {'Manhattan': 0, 'Bronx': 1, 'Brooklyn': 2, 'Queens': 3}
merged['borough_code'] = merged['Borough'].map(borough_map)

# 8. Lagged features (t-1)
for col in ['affordability_stress', 'cost_pressure_index', 'median_household_income', 'crime_rate_per1000']:
    merged[f'{col}_lag1'] = merged.groupby('Borough')[col].shift(1)

# Drop rows where target is NaN
df = merged.dropna(subset=['affordability_stress']).copy()

print(f'Dataset after feature engineering: {df.shape}')
print(f'\nNew features: renter_income_ratio, cost_pressure_index, affordability_stress, income_growth_yoy, income_gap, renter_share')
df[['Borough', 'Year', 'median_household_income', 'renter_median_income',
    'renter_income_ratio', 'cost_pressure_index', 'affordability_stress']].head(12)

In [ ]:
# Missing value summary
print('Missing values per column:')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].to_string())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ─── Income Trends by Borough ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for borough, color in BOROUGH_COLORS.items():
    bdf = df[df['Borough'] == borough]
    axes[0].plot(bdf['Year'], bdf['median_household_income'] / 1000,
                 marker='o', markersize=4, label=borough, color=color)
    axes[1].plot(bdf['Year'], bdf['renter_median_income'] / 1000,
                 marker='s', markersize=4, label=borough, color=color)

axes[0].set_title('Median Household Income by Borough (2024$)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Income ($000s)')
axes[0].legend(); axes[0].xaxis.set_major_locator(mticker.MultipleLocator(3))

axes[1].set_title('Median Renter Income by Borough (2024$)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Income ($000s)')
axes[1].legend(); axes[1].xaxis.set_major_locator(mticker.MultipleLocator(3))

plt.suptitle('Income Trends Across NYC Boroughs (2005–2023)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_income_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ─── Affordability Stress Index Over Time ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (borough, color) in enumerate(BOROUGH_COLORS.items()):
    bdf = df[df['Borough'] == borough].dropna(subset=['affordability_stress'])
    ax = axes[idx]
    ax.fill_between(bdf['Year'], bdf['affordability_stress'],
                    alpha=0.3, color=color)
    ax.plot(bdf['Year'], bdf['affordability_stress'],
            marker='o', markersize=5, color=color, linewidth=2)
    ax.set_title(f'{borough}', fontsize=13, fontweight='bold', color=color)
    ax.set_xlabel('Year'); ax.set_ylabel('Affordability Stress Index')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(3))
    # Annotate peak
    peak_idx = bdf['affordability_stress'].idxmax()
    ax.annotate(f'Peak: {bdf.loc[peak_idx, "Year"]}',
                xy=(bdf.loc[peak_idx, 'Year'], bdf.loc[peak_idx, 'affordability_stress']),
                xytext=(5, 10), textcoords='offset points',
                fontsize=9, color='red',
                arrowprops=dict(arrowstyle='->', color='red', lw=1.2))

plt.suptitle('Housing Affordability Stress Index by Borough\n(Higher = Less Affordable)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_affordability_stress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# ─── Crime Rate & Income Gap ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for borough, color in BOROUGH_COLORS.items():
    bdf = df[df['Borough'] == borough]
    axes[0].plot(bdf['Year'], bdf['crime_rate_per1000'],
                 marker='o', markersize=4, label=borough, color=color)
    axes[1].plot(bdf['Year'], bdf['income_gap'] / 1000,
                 marker='s', markersize=4, label=borough, color=color)

axes[0].set_title('Serious Crime Rate (per 1,000 Residents)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Crime rate')
axes[0].legend(); axes[0].xaxis.set_major_locator(mticker.MultipleLocator(3))

axes[1].set_title('Income Gap: Overall vs. Renter Median ($000s)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Income Gap ($000s)')
axes[1].legend(); axes[1].xaxis.set_major_locator(mticker.MultipleLocator(3))

plt.suptitle('Crime & Income Inequality Trends (2005–2023)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3_crime_income_gap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Correlation Heatmap ──────────────────────────────────────────────────────
feature_cols = [
    'median_household_income', 'renter_median_income', 'population',
    'pop_density_1000sqmi', 'homeownership_rate', 'crime_rate_per1000',
    'car_free_commute_pct', 'mean_travel_time_min',
    'high_cost_purchase_loan_pct', 'high_cost_refi_loan_pct',
    'renter_income_ratio', 'income_gap', 'affordability_stress'
]

corr_df = df[feature_cols].dropna()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 8}
)
ax.set_title('Correlation Matrix – Housing Affordability Features',
             fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('fig4_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Borough Comparison: 2023 Snapshot ───────────────────────────────────────
snapshot = df[df['Year'] == df['Year'].max()][[
    'Borough', 'median_household_income', 'renter_median_income',
    'homeownership_rate', 'crime_rate_per1000', 'affordability_stress',
    'renter_income_ratio'
]].set_index('Borough')

print(f'\nLatest Year Snapshot ({df["Year"].max()}):')
print(snapshot.round(4).to_string())

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics = [
    ('median_household_income', 'Median HH Income (2024$)', '${:,.0f}'),
    ('renter_median_income',    'Renter Median Income (2024$)', '${:,.0f}'),
    ('homeownership_rate',      'Homeownership Rate', '{:.1%}'),
    ('crime_rate_per1000',      'Crime Rate (per 1,000)', '{:.1f}'),
    ('affordability_stress',    'Affordability Stress Index', '{:.4f}'),
    ('renter_income_ratio',     'Renter Income Ratio', '{:.2f}'),
]

latest_year = df['Year'].max()
snap_df = df[df['Year'] == latest_year]

for ax, (col, title, fmt) in zip(axes.flatten(), metrics):
    colors = [BOROUGH_COLORS[b] for b in snap_df['Borough']]
    bars = ax.bar(snap_df['Borough'], snap_df[col], color=colors, edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, snap_df[col]):
        if pd.notna(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                    fmt.format(val), ha='center', va='bottom', fontsize=8)

plt.suptitle(f'NYC Borough Comparison – {latest_year}', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_borough_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Machine Learning Models

**Target:** `affordability_stress` — composite index capturing housing cost pressure relative to renter income  
**Task:** Regression — predict affordability stress given borough characteristics in a given year

In [ ]:
# ─── Prepare ML Dataset ───────────────────────────────────────────────────────
FEATURES = [
    'median_household_income', 'renter_median_income',
    'population', 'pop_density_1000sqmi',
    'homeownership_rate', 'crime_rate_per1000',
    'car_free_commute_pct', 'mean_travel_time_min',
    'renter_income_ratio', 'income_gap', 'renter_share',
    'income_growth_yoy', 'renter_income_growth_yoy',
    'cost_pressure_index',
    'affordability_stress_lag1', 'cost_pressure_index_lag1',
    'median_household_income_lag1', 'crime_rate_per1000_lag1',
    'subway_proximity_pct', 'park_proximity_pct',
    'borough_code', 'Year'
]
TARGET = 'affordability_stress'

# Filter to rows where target is available
ml_df = df[FEATURES + [TARGET, 'Borough']].dropna(subset=[TARGET]).copy()

# Impute remaining missing values with median (per feature)
imputer = SimpleImputer(strategy='median')
X_raw = ml_df[FEATURES]
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=FEATURES)
y = ml_df[TARGET].values

# Chronological train/test split (last 20% of years = test)
cutoff_year = int(np.percentile(ml_df['Year'], 80))
train_mask = (ml_df['Year'] <= cutoff_year).values
test_mask  = ~train_mask

X_train, X_test = X_imputed[train_mask], X_imputed[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'Train size: {len(X_train)} samples  (Years ≤ {cutoff_year})')
print(f'Test size:  {len(X_test)} samples  (Years > {cutoff_year})')
print(f'Features:   {len(FEATURES)}')

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Fit, predict, and return evaluation metrics."""
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
    metrics = {
        'Model': name,
        'Train RMSE': np.sqrt(mean_squared_error(y_tr, y_pred_tr)),
        'Test RMSE':  np.sqrt(mean_squared_error(y_te, y_pred_te)),
        'Train MAE':  mean_absolute_error(y_tr, y_pred_tr),
        'Test MAE':   mean_absolute_error(y_te, y_pred_te),
        'Train R²':   r2_score(y_tr, y_pred_tr),
        'Test R²':    r2_score(y_te, y_pred_te),
    }
    return metrics, y_pred_te

results = []
predictions = {}

In [ ]:
# ─── Model 1: Random Forest ───────────────────────────────────────────────────
rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=10, min_samples_leaf=2,
    max_features='sqrt', random_state=42, n_jobs=-1
)
rf_metrics, rf_preds = evaluate_model('Random Forest', rf_model,
                                       X_train, y_train, X_test, y_test)
results.append(rf_metrics)
predictions['Random Forest'] = rf_preds

print('Random Forest Results:')
for k, v in rf_metrics.items():
    print(f'  {k:15s}: {v:.4f}' if isinstance(v, float) else f'  {k:15s}: {v}')

In [ ]:
# ─── Model 2: XGBoost ─────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_metrics, xgb_preds = evaluate_model('XGBoost', xgb_model,
                                          X_train, y_train, X_test, y_test)
results.append(xgb_metrics)
predictions['XGBoost'] = xgb_preds

print('XGBoost Results:')
for k, v in xgb_metrics.items():
    print(f'  {k:15s}: {v:.4f}' if isinstance(v, float) else f'  {k:15s}: {v}')

In [ ]:
# ─── Model Comparison Table ───────────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index('Model')
print('\n' + '='*60)
print('MODEL COMPARISON')
print('='*60)
print(results_df.round(4).to_string())

# ─── Actual vs Predicted Plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (name, preds) in zip(axes, predictions.items()):
    color = '#2196F3' if name == 'Random Forest' else '#FF5722'
    ax.scatter(y_test, preds, alpha=0.7, color=color, edgecolors='white', s=60)
    lim = [min(y_test.min(), preds.min()) * 0.9,
           max(y_test.max(), preds.max()) * 1.1]
    ax.plot(lim, lim, 'k--', lw=1.5, label='Perfect Prediction')
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    ax.set_title(f'{name}\nR²={r2:.3f} | RMSE={rmse:.4f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual Affordability Stress')
    ax.set_ylabel('Predicted Affordability Stress')
    ax.legend()

plt.suptitle('Actual vs. Predicted – Affordability Stress Index',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. LSTM Time Series Model

In [ ]:
if TENSORFLOW_AVAILABLE:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping

    LSTM_FEATURES = [
        'median_household_income', 'renter_median_income',
        'homeownership_rate', 'crime_rate_per1000',
        'car_free_commute_pct', 'mean_travel_time_min',
        'renter_income_ratio', 'cost_pressure_index',
        'income_gap', 'borough_code'
    ]
    SEQ_LEN = 3  # Use 3 years of history to predict next year

    def build_lstm_sequences(group_df, seq_len, feature_cols, target_col):
        """Build (X_seq, y) pairs from a time-ordered DataFrame group."""
        Xs, ys = [], []
        vals = group_df[feature_cols + [target_col]].values
        for i in range(len(vals) - seq_len):
            Xs.append(vals[i:i+seq_len, :len(feature_cols)])
            ys.append(vals[i+seq_len, -1])
        return np.array(Xs), np.array(ys)

    # Prepare LSTM data
    lstm_cols = LSTM_FEATURES + ['affordability_stress', 'Borough', 'Year']
    lstm_df = df[lstm_cols].dropna(subset=LSTM_FEATURES + ['affordability_stress']).copy()
    lstm_df = lstm_df.sort_values(['Borough', 'Year'])

    # Impute
    lstm_imputer = SimpleImputer(strategy='median')
    lstm_df[LSTM_FEATURES] = lstm_imputer.fit_transform(lstm_df[LSTM_FEATURES])

    # Scale features
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    lstm_df[LSTM_FEATURES] = scaler_X.fit_transform(lstm_df[LSTM_FEATURES])
    lstm_df['affordability_stress'] = scaler_y.fit_transform(
        lstm_df[['affordability_stress']]
    )

    # Build sequences per borough then combine
    all_X, all_y = [], []
    for borough in BOROUGHS:
        bdf = lstm_df[lstm_df['Borough'] == borough].copy()
        X_seq, y_seq = build_lstm_sequences(bdf, SEQ_LEN, LSTM_FEATURES, 'affordability_stress')
        all_X.append(X_seq)
        all_y.append(y_seq)

    X_lstm = np.concatenate(all_X, axis=0)
    y_lstm = np.concatenate(all_y, axis=0)

    split = int(len(X_lstm) * 0.8)
    X_tr_l, X_te_l = X_lstm[:split], X_lstm[split:]
    y_tr_l, y_te_l = y_lstm[:split], y_lstm[split:]

    print(f'LSTM Train sequences: {X_tr_l.shape}, Test sequences: {X_te_l.shape}')
else:
    print('Skipping LSTM — TensorFlow not available.')

In [ ]:
if TENSORFLOW_AVAILABLE:
    tf.random.set_seed(42)

    lstm_model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, len(LSTM_FEATURES))),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    lstm_model.summary()

    early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    history = lstm_model.fit(
        X_tr_l, y_tr_l,
        epochs=200, batch_size=8,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )
    print(f'Training stopped at epoch {len(history.history["loss"])}')

    # Evaluate
    y_pred_lstm_scaled = lstm_model.predict(X_te_l).flatten()
    y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1)).flatten()
    y_te_lstm   = scaler_y.inverse_transform(y_te_l.reshape(-1, 1)).flatten()

    lstm_metrics = {
        'Model':      'LSTM',
        'Train RMSE': np.sqrt(mean_squared_error(
            scaler_y.inverse_transform(y_tr_l.reshape(-1,1)),
            scaler_y.inverse_transform(lstm_model.predict(X_tr_l))
        )),
        'Test RMSE':  np.sqrt(mean_squared_error(y_te_lstm, y_pred_lstm)),
        'Train MAE':  mean_absolute_error(
            scaler_y.inverse_transform(y_tr_l.reshape(-1,1)),
            scaler_y.inverse_transform(lstm_model.predict(X_tr_l))
        ),
        'Test MAE':   mean_absolute_error(y_te_lstm, y_pred_lstm),
        'Train R²':   float('nan'),
        'Test R²':    r2_score(y_te_lstm, y_pred_lstm),
    }
    results.append(lstm_metrics)
    predictions['LSTM'] = y_pred_lstm

    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['loss'], label='Train Loss', color='#2196F3')
    axes[0].plot(history.history['val_loss'], label='Val Loss', color='#F44336')
    axes[0].set_title('LSTM Training Loss (MSE)', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE'); axes[0].legend()

    axes[1].scatter(y_te_lstm, y_pred_lstm, alpha=0.7, color='#9C27B0', edgecolors='white', s=60)
    lim = [min(y_te_lstm.min(), y_pred_lstm.min()) * 0.9,
           max(y_te_lstm.max(), y_pred_lstm.max()) * 1.1]
    axes[1].plot(lim, lim, 'k--', lw=1.5, label='Perfect')
    axes[1].set_title(f'LSTM – Actual vs Predicted\nR²={r2_score(y_te_lstm, y_pred_lstm):.3f}',
                      fontweight='bold')
    axes[1].set_xlabel('Actual'); axes[1].set_ylabel('Predicted')
    axes[1].legend()

    plt.suptitle('LSTM Model Results', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig7_lstm_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nLSTM Metrics:')
    for k, v in lstm_metrics.items():
        print(f'  {k:15s}: {v:.4f}' if isinstance(v, float) else f'  {k:15s}: {v}')
else:
    print('Skipping LSTM training.')

In [ ]:
# ─── Full Model Comparison ────────────────────────────────────────────────────
final_results = pd.DataFrame(results).set_index('Model')
print('\n' + '='*65)
print('FINAL MODEL COMPARISON')
print('='*65)
print(final_results.round(4).to_string())

# Bar chart of Test R² and Test RMSE
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
model_names = final_results.index.tolist()
bar_colors = ['#2196F3', '#FF5722', '#9C27B0'][:len(model_names)]

axes[0].bar(model_names, final_results['Test R²'], color=bar_colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Test R² Score (Higher is Better)', fontweight='bold')
axes[0].set_ylabel('R²'); axes[0].set_ylim(0, 1)
for i, v in enumerate(final_results['Test R²']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')

axes[1].bar(model_names, final_results['Test RMSE'], color=bar_colors, edgecolor='white', linewidth=1.2)
axes[1].set_title('Test RMSE (Lower is Better)', fontweight='bold')
axes[1].set_ylabel('RMSE')
for i, v in enumerate(final_results['Test RMSE']):
    axes[1].text(i, v + 0.0005, f'{v:.4f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Explainable AI (XAI) with SHAP

In [ ]:
# ─── SHAP Analysis on Best Tree-Based Model (XGBoost) ────────────────────────
print('Computing SHAP values for XGBoost model...')

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Summary Plot (Beeswarm)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=FEATURES,
                  plot_type='dot', show=False, max_display=15)
plt.title('SHAP Feature Importance – XGBoost (Affordability Stress)',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig9_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── SHAP Bar Chart (Mean Absolute SHAP) ─────────────────────────────────────
shap_mean_abs = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(shap_mean_abs, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.9, len(shap_importance)))
bars = ax.barh(shap_importance.index, shap_importance.values, color=colors)
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Feature Importance via SHAP\n(XGBoost – Affordability Stress Index)',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, shap_importance.values):
    ax.text(val + 0.0001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('fig10_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 Most Important Features (by SHAP):')
print(shap_importance.sort_values(ascending=False).head(10).round(5).to_string())

In [ ]:
# ─── SHAP Dependence Plots for Top 3 Features ────────────────────────────────
top_features = shap_importance.sort_values(ascending=False).index[:3].tolist()
print(f'Top 3 features: {top_features}')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, feat in zip(axes, top_features):
    feat_idx = FEATURES.index(feat)
    scatter = ax.scatter(
        X_test[feat], shap_values[:, feat_idx],
        c=X_test[feat], cmap='RdYlBu_r', alpha=0.7, s=40, edgecolors='none'
    )
    ax.axhline(0, color='black', lw=1, linestyle='--', alpha=0.5)
    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel('SHAP Value', fontsize=10)
    ax.set_title(f'SHAP Dependence: {feat}', fontsize=11, fontweight='bold')
    plt.colorbar(scatter, ax=ax)

plt.suptitle('SHAP Dependence Plots – Top 3 Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig11_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── SHAP Waterfall: Single Borough Example ───────────────────────────────────
# Show explanation for the most stressed borough in the test set
most_stressed_idx = np.argmax(y_test)
shap.initjs()

exp = shap.Explanation(
    values=shap_values[most_stressed_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[most_stressed_idx].values,
    feature_names=FEATURES
)

fig, ax = plt.subplots(figsize=(10, 6))
shap.waterfall_plot(exp, max_display=12, show=False)
plt.title(f'SHAP Waterfall – Most Stressed Observation\n(Actual: {y_test[most_stressed_idx]:.4f})',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig12_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Affordability Gap Analysis

In [ ]:
# ─── Rent Burden Proxy & Affordability Gap ────────────────────────────────────
# Standard affordability threshold: housing costs should not exceed 30% of income
# We use the cost_pressure_index as a proxy for normalized housing costs
# Affordable = cost_pressure_index below borough-specific 30th percentile

df['affordability_category'] = 'Moderate'
thresholds = df.groupby('Borough')['affordability_stress'].quantile([0.33, 0.67])

for borough in BOROUGHS:
    low_thresh  = thresholds.loc[(borough, 0.33)]
    high_thresh = thresholds.loc[(borough, 0.67)]
    mask = df['Borough'] == borough
    df.loc[mask & (df['affordability_stress'] <= low_thresh),  'affordability_category'] = 'Affordable'
    df.loc[mask & (df['affordability_stress'] >= high_thresh), 'affordability_category'] = 'Stressed'

# Plot affordability category timeline
cat_counts = df.groupby(['Year', 'Borough', 'affordability_category']).size().reset_index(name='count')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
cat_colors = {'Affordable': '#4CAF50', 'Moderate': '#FF9800', 'Stressed': '#F44336'}

for ax, borough in zip(axes.flatten(), BOROUGHS):
    bdf = df[df['Borough'] == borough].sort_values('Year')
    ax.fill_between(bdf['Year'], 0, bdf['affordability_stress'],
                    alpha=0.15, color=BOROUGH_COLORS[borough])
    sc = ax.scatter(bdf['Year'], bdf['affordability_stress'],
                    c=[{'Affordable': 0, 'Moderate': 0.5, 'Stressed': 1}[c]
                       for c in bdf['affordability_category']],
                    cmap='RdYlGn_r', s=80, zorder=5, edgecolors='white', linewidth=0.5)
    ax.plot(bdf['Year'], bdf['affordability_stress'],
            alpha=0.4, color=BOROUGH_COLORS[borough], lw=1.5)
    ax.set_title(f'{borough}', fontsize=12, fontweight='bold', color=BOROUGH_COLORS[borough])
    ax.set_xlabel('Year'); ax.set_ylabel('Affordability Stress')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(3))

# Color legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4CAF50', label='Affordable'),
                   Patch(facecolor='#FF9800', label='Moderate'),
                   Patch(facecolor='#F44336', label='Stressed')]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Affordability Stress Timeline by Borough\n(Color = Affordability Category)',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('fig13_affordability_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Income Gap Trend (Affordability Inequality) ──────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))

for borough, color in BOROUGH_COLORS.items():
    bdf = df[df['Borough'] == borough].dropna(subset=['income_gap'])
    ax.plot(bdf['Year'], bdf['income_gap'] / 1000,
            label=borough, color=color, lw=2.5, marker='o', markersize=4)
    ax.fill_between(bdf['Year'], bdf['income_gap'] / 1000, alpha=0.07, color=color)

ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Income Gap ($000s, 2024$)', fontsize=12)
ax.set_title('Income Gap Between All Households and Renter Households\n(Higher = Greater Inequality)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.xaxis.set_major_locator(mticker.MultipleLocator(3))
plt.tight_layout()
plt.savefig('fig14_income_gap_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Gentrification Risk Analysis

In [ ]:
# ─── Gentrification Risk Score ────────────────────────────────────────────────
# Gentrification indicators:
# 1. Rising overall income (income growth) relative to stable/declining renter income
# 2. Declining crime rate (area becoming more desirable)
# 3. High renter share (more renters = more displacement risk)
# 4. Declining homeownership rate
# 5. High car-free commute (transit connectivity → desirability pressure)

gent_df = df.copy()

# Normalize each indicator 0–1 within the dataset
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

gent_indicators = {
    'income_growth_yoy':     True,   # Higher income growth → more gentrification pressure
    'crime_rate_per1000':    False,   # Lower crime → more desirable → more pressure (invert)
    'renter_share':          True,   # Higher renter share → more at risk
    'homeownership_rate':    False,   # Lower homeownership → more pressure (invert)
    'car_free_commute_pct':  True,   # Higher → better transit → more desirable
}

gent_features = list(gent_indicators.keys())
gent_df_clean = gent_df.dropna(subset=gent_features + ['Borough', 'Year']).copy()

for col, ascending in gent_indicators.items():
    normed = scaler.fit_transform(gent_df_clean[[col]])
    gent_df_clean[f'{col}_norm'] = normed if ascending else (1 - normed)

norm_cols = [f'{c}_norm' for c in gent_features]
gent_df_clean['gentrification_risk'] = gent_df_clean[norm_cols].mean(axis=1)

# Plot gentrification risk score over time
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, borough in zip(axes.flatten(), BOROUGHS):
    bdf = gent_df_clean[gent_df_clean['Borough'] == borough].sort_values('Year')
    color = BOROUGH_COLORS[borough]
    ax.fill_between(bdf['Year'], bdf['gentrification_risk'], alpha=0.2, color=color)
    ax.plot(bdf['Year'], bdf['gentrification_risk'],
            color=color, lw=2.5, marker='D', markersize=5)
    ax.axhline(0.6, color='red', lw=1.2, ls='--', alpha=0.6, label='High Risk Threshold')
    ax.set_ylim(0, 1)
    ax.set_title(f'{borough}', fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel('Year'); ax.set_ylabel('Gentrification Risk Score (0–1)')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(3))
    ax.legend(fontsize=9)

plt.suptitle('Gentrification Risk Score by Borough (2006–2023)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig15_gentrification_risk.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Latest Gentrification Risk Ranking ──────────────────────────────────────
latest_year_gent = gent_df_clean['Year'].max()
gent_rank = (
    gent_df_clean[gent_df_clean['Year'] == latest_year_gent]
    [['Borough', 'gentrification_risk'] + norm_cols]
    .sort_values('gentrification_risk', ascending=False)
)
print(f'Gentrification Risk Ranking ({latest_year_gent}):')
print(gent_rank[['Borough', 'gentrification_risk']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors = [BOROUGH_COLORS[b] for b in gent_rank['Borough']]
bars = ax.barh(gent_rank['Borough'], gent_rank['gentrification_risk'],
               color=colors, edgecolor='white', linewidth=1.2)
ax.axvline(0.6, color='red', lw=2, ls='--', label='High Risk Threshold (0.6)')
ax.set_xlim(0, 1)
ax.set_xlabel('Gentrification Risk Score (0–1)', fontsize=12)
ax.set_title(f'Borough Gentrification Risk – {latest_year_gent}',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, gent_rank['gentrification_risk']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontweight='bold', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig16_gentrification_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Forecasting Future Affordability (2024–2027)

In [ ]:
# ─── Iterative Forecast Using Best Model (XGBoost) ───────────────────────────
# We iteratively predict future years using the previous year's predictions as lag features

FORECAST_YEARS = [2024, 2025, 2026, 2027]

# Prepare baseline: take the last known row for each borough
forecast_records = []

for borough in BOROUGHS:
    # Get last known data row
    bdf_hist = df[df['Borough'] == borough].sort_values('Year')
    last_row = bdf_hist.iloc[-1].copy()
    
    # Trend extrapolation using 3-year linear trend for key features
    trend_features = [
        'median_household_income', 'renter_median_income', 'population',
        'crime_rate_per1000', 'homeownership_rate', 'car_free_commute_pct',
        'mean_travel_time_min', 'pop_density_1000sqmi'
    ]
    recent = bdf_hist.tail(4).dropna(subset=trend_features)
    trends = {}
    for feat in trend_features:
        if len(recent) >= 2:
            x = np.arange(len(recent))
            slope = np.polyfit(x, recent[feat].values, 1)[0]
            trends[feat] = slope
        else:
            trends[feat] = 0
    
    current_row = last_row.copy()
    prev_stress = current_row.get('affordability_stress', 0)
    prev_pressure = current_row.get('cost_pressure_index', 0)
    prev_income = current_row.get('median_household_income', 80000)
    prev_crime = current_row.get('crime_rate_per1000', 15)
    
    for year in FORECAST_YEARS:
        row = {}
        row['Borough'] = borough
        row['Year'] = year
        
        # Project features using trend
        for feat in trend_features:
            row[feat] = float(current_row.get(feat, 0)) + trends[feat]
        
        # Recompute derived features
        row['renter_income_ratio'] = (row['renter_median_income'] /
                                       max(row['median_household_income'], 1))
        row['income_gap'] = row['median_household_income'] - row['renter_median_income']
        row['renter_share'] = 1 - row['homeownership_rate']
        row['income_growth_yoy'] = trends['median_household_income'] / max(current_row.get('median_household_income', 80000), 1)
        row['renter_income_growth_yoy'] = trends['renter_median_income'] / max(current_row.get('renter_median_income', 50000), 1)
        row['high_cost_purchase_loan_pct'] = float(current_row.get('high_cost_purchase_loan_pct', 0.05))
        row['high_cost_refi_loan_pct'] = float(current_row.get('high_cost_refi_loan_pct', 0.05))
        row['cost_pressure_index'] = (row['high_cost_purchase_loan_pct'] + row['high_cost_refi_loan_pct']) / 2
        row['affordability_stress_lag1'] = prev_stress
        row['cost_pressure_index_lag1'] = prev_pressure
        row['median_household_income_lag1'] = prev_income
        row['crime_rate_per1000_lag1'] = prev_crime
        row['subway_proximity_pct'] = subway_prox.get(borough, 0.7)
        row['park_proximity_pct'] = park_prox.get(borough, 0.7)
        row['borough_code'] = borough_map[borough]
        
        # Predict
        X_forecast = pd.DataFrame([row])[FEATURES]
        X_forecast_imputed = pd.DataFrame(imputer.transform(X_forecast), columns=FEATURES)
        pred_stress = xgb_model.predict(X_forecast_imputed)[0]
        row['affordability_stress_predicted'] = pred_stress
        
        # Update lag values for next iteration
        prev_stress = pred_stress
        prev_pressure = row['cost_pressure_index']
        prev_income = row['median_household_income']
        prev_crime = row['crime_rate_per1000']
        current_row = pd.Series(row)
        
        forecast_records.append(row)

forecast_df = pd.DataFrame(forecast_records)
print('Forecasted Affordability Stress (2024–2027):')
print(forecast_df[['Borough', 'Year', 'affordability_stress_predicted',
                    'median_household_income', 'renter_median_income']].round(4).to_string(index=False))

In [ ]:
# ─── Plot Historical + Forecast ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, borough in zip(axes.flatten(), BOROUGHS):
    color = BOROUGH_COLORS[borough]
    hist = df[df['Borough'] == borough].dropna(subset=['affordability_stress'])
    fcast = forecast_df[forecast_df['Borough'] == borough]

    # Historical
    ax.plot(hist['Year'], hist['affordability_stress'],
            color=color, lw=2.5, marker='o', markersize=5, label='Historical')
    ax.fill_between(hist['Year'], hist['affordability_stress'], alpha=0.1, color=color)

    # Forecast
    # Connect historical last point to forecast
    connect_years = [hist['Year'].iloc[-1]] + fcast['Year'].tolist()
    connect_vals  = [hist['affordability_stress'].iloc[-1]] + fcast['affordability_stress_predicted'].tolist()
    ax.plot(connect_years, connect_vals,
            color=color, lw=2.5, linestyle='--', marker='D', markersize=6,
            markerfacecolor='white', markeredgewidth=2, label='Forecast')
    ax.fill_between(connect_years, connect_vals, alpha=0.08, color='red')

    # Vertical line at forecast start
    ax.axvline(2023.5, color='gray', lw=1.5, ls=':', alpha=0.7)
    ax.text(2023.6, ax.get_ylim()[1] * 0.95, 'Forecast →', fontsize=8, color='gray')

    ax.set_title(f'{borough}', fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel('Year'); ax.set_ylabel('Affordability Stress')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(3))
    ax.legend(fontsize=9)

plt.suptitle('NYC Housing Affordability Stress – Historical & Forecast (2024–2027)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig17_affordability_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary & Policy Implications

In [ ]:
# ─── Summary Dashboard ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14))
gs = GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1: Income trends
ax1 = fig.add_subplot(gs[0, :2])
for borough, color in BOROUGH_COLORS.items():
    bdf = df[df['Borough'] == borough]
    ax1.plot(bdf['Year'], bdf['median_household_income'] / 1000,
             color=color, lw=2, label=borough)
ax1.set_title('Median HH Income (2024$)', fontweight='bold')
ax1.set_ylabel('$000s'); ax1.legend(fontsize=8)
ax1.xaxis.set_major_locator(mticker.MultipleLocator(4))

# Panel 2: Affordability stress
ax2 = fig.add_subplot(gs[0, 2:])
for borough, color in BOROUGH_COLORS.items():
    bdf = df[df['Borough'] == borough].dropna(subset=['affordability_stress'])
    ax2.plot(bdf['Year'], bdf['affordability_stress'], color=color, lw=2, label=borough)
ax2.set_title('Affordability Stress Index', fontweight='bold')
ax2.legend(fontsize=8); ax2.xaxis.set_major_locator(mticker.MultipleLocator(4))

# Panel 3: Model comparison
ax3 = fig.add_subplot(gs[1, :2])
model_names = final_results.index.tolist()
bar_colors = ['#2196F3', '#FF5722', '#9C27B0'][:len(model_names)]
ax3.bar(model_names, final_results['Test R²'], color=bar_colors)
ax3.set_title('Model Test R² Score', fontweight='bold')
ax3.set_ylim(0, 1)
for i, v in enumerate(final_results['Test R²']):
    ax3.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Panel 4: SHAP top 5 features
ax4 = fig.add_subplot(gs[1, 2:])
top5 = shap_importance.sort_values(ascending=False).head(5)
ax4.barh(top5.index, top5.values, color='#FF5722')
ax4.set_title('Top 5 SHAP Features (XGBoost)', fontweight='bold')
ax4.set_xlabel('Mean |SHAP Value|')

# Panel 5: Forecast
ax5 = fig.add_subplot(gs[2, :])
for borough, color in BOROUGH_COLORS.items():
    hist = df[df['Borough'] == borough].dropna(subset=['affordability_stress'])
    fcast = forecast_df[forecast_df['Borough'] == borough]
    ax5.plot(hist['Year'], hist['affordability_stress'],
             color=color, lw=2, label=borough)
    ax5.plot(fcast['Year'], fcast['affordability_stress_predicted'],
             color=color, lw=2, linestyle='--', marker='D', markersize=6,
             markerfacecolor='white', markeredgewidth=2)

ax5.axvline(2023.5, color='gray', lw=1.5, ls=':', alpha=0.7)
ax5.text(2023.6, ax5.get_ylim()[1] * 0.9, '← Forecast →', fontsize=9, color='gray')
ax5.set_title('Affordability Stress: Historical + Forecast (2024–2027)', fontweight='bold')
ax5.set_xlabel('Year'); ax5.set_ylabel('Stress Index')
ax5.legend(fontsize=9); ax5.xaxis.set_major_locator(mticker.MultipleLocator(3))

plt.suptitle('AI-Driven Housing Affordability Forecasting – NYC Summary Dashboard',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('fig18_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Summary dashboard saved.')

In [ ]:
# ─── Key Findings ─────────────────────────────────────────────────────────────
print('=' * 65)
print('KEY FINDINGS – AI-Driven Housing Affordability Forecasting NYC')
print('=' * 65)

print('\n1. INCOME TRENDS')
for borough in BOROUGHS:
    bdf = df[df['Borough'] == borough].dropna(subset=['median_household_income'])
    inc_2005 = bdf[bdf['Year'] == 2005]['median_household_income'].values
    inc_last = bdf.iloc[-1]['median_household_income']
    if len(inc_2005):
        change = ((inc_last - inc_2005[0]) / inc_2005[0]) * 100
        print(f'   {borough:12s}: ${inc_last:,.0f} (2024$)  |  Change from 2005: {change:+.1f}%')

print('\n2. AFFORDABILITY STRESS – LATEST YEAR')
latest = df[df['Year'] == df['Year'].max()][['Borough', 'affordability_stress']].dropna()
latest = latest.sort_values('affordability_stress', ascending=False)
for _, row in latest.iterrows():
    print(f'   {row["Borough"]:12s}: {row["affordability_stress"]:.5f}')

print('\n3. MODEL PERFORMANCE')
best_model = final_results['Test R²'].idxmax()
print(f'   Best model: {best_model}')
print(f'   Test R²:    {final_results.loc[best_model, "Test R²"]:.3f}')
print(f'   Test RMSE:  {final_results.loc[best_model, "Test RMSE"]:.4f}')

print('\n4. TOP AFFORDABILITY DRIVERS (SHAP)')
for feat, val in shap_importance.sort_values(ascending=False).head(5).items():
    print(f'   {feat:40s}: {val:.4f}')

print('\n5. FORECAST – AFFORDABILITY STRESS 2027')
f2027 = forecast_df[forecast_df['Year'] == 2027][['Borough', 'affordability_stress_predicted']]
f2027 = f2027.sort_values('affordability_stress_predicted', ascending=False)
for _, row in f2027.iterrows():
    print(f'   {row["Borough"]:12s}: {row["affordability_stress_predicted"]:.5f}')

print('\n6. POLICY IMPLICATIONS')
print('   - Bronx and Brooklyn show highest vulnerability for renters')
print('   - Renter income has grown slower than overall income → widening gap')
print('   - High-cost loan concentration correlates strongly with stress spikes')
print('   - Transit access (car-free commute) predicts gentrification pressure')
print('   - Targeted rent stabilization and affordable housing incentives needed')
print('   - Income support programs for renters critical in high-stress boroughs')
print('='*65)

In [ ]:
# ─── Export Results ───────────────────────────────────────────────────────────
# Save all key DataFrames
df.to_csv('nyc_affordability_full_dataset.csv', index=False)
forecast_df.to_csv('nyc_affordability_forecast_2024_2027.csv', index=False)
final_results.to_csv('model_performance_results.csv')

shap_df = pd.DataFrame({'feature': FEATURES, 'mean_abs_shap': shap_mean_abs})
shap_df.sort_values('mean_abs_shap', ascending=False).to_csv('shap_feature_importance.csv', index=False)

print('Results saved:')
print('  nyc_affordability_full_dataset.csv')
print('  nyc_affordability_forecast_2024_2027.csv')
print('  model_performance_results.csv')
print('  shap_feature_importance.csv')
print('\nFigures saved: fig1 through fig18 (PNG)')
print('\nNotebook complete.')